# Workbook 3 · Technology and architecture: better cells, redundancy, and model switches

Requirements say *what* the aircraft must do; assumptions say *with what*. This workbook changes three kinds of assumption and shows how to tell a
technology gain from a modelling artefact:

1. a **technology** assumption: a more power-dense battery cell;
2. an **architecture** choice: drop the redundancy (lanes, buses, strings and their failure hovers);
3. a **model** switch: turn the thermal model off, and see why the answer goes the "wrong" way.

**Run time:** about 2 minutes (three solves).

In [1]:
import sys
from pathlib import Path
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(repo_root / "tutorials"))
from sizing_workbench import *
from dataclasses import replace
base = baseline_variant()
a = base.assumptions
print(f"battery: {a.count_series_battery} cells in series, power-density factor {a.factor_power_density_battery}, "
      f"end-of-life capacity {a.factor_capacity_ageing_battery}, resistance x{a.factor_resistance_ageing_battery}")
print(f"redundancy {a.redundancy}: {a.count_lanes_motor} lanes per rotor, {a.count_buses} buses, {a.count_strings_battery} battery strings; "
      f"failure hovers: lane {a.failure_lane_out}, bus {a.failure_bus_out}, string {a.failure_string_out}")
print(f"thermal model {a.thermal_model}")

battery: 210 cells in series, power-density factor 5.0, end-of-life capacity 0.8, resistance x1.5
redundancy True: 2 lanes per rotor, 2 buses, 2 battery strings; failure hovers: lane True, bus True, string True
thermal model True


## 1. A more power-dense cell

`factor_power_density_battery` scales the 50G cell's resistance down and its current rating up (tutorial 07): 5 in the baseline, an explicit assumption about
next-generation cells. Since voltage sag in the engine-out hover sizes the pack, a lower-resistance cell should shrink it a lot. Try 8:

In [2]:
cell8 = size("cell power density 8", assumptions=replace(a, factor_power_density_battery=8.0))
compare(base, cell8)
weight_changes(base, cell8)
check("a more power-dense cell shrinks the battery", cell8.result.energy_battery_kWh < base.result.energy_battery_kWh)

[cell power density 8: sized in 50 s from 'baseline': 14,393 lb]
                                            baseline    cell power density 8
take-off weight [lb]                          15,134          14,393 (-4.9%)
empty weight [lb]                             11,087          10,400 (-6.2%)
payload [lb]                                   1,984           1,984 (+0.0%)
fuel [lb]                                      2,063           2,009 (-2.6%)
battery [kWh]                                   69.5           55.3 (-20.5%)
wing area [m2]                                 23.24           22.33 (-3.9%)
span [m]                                       11.93           11.69 (-2.0%)
rotor radius [m]                               4.825           4.706 (-2.5%)
disk loading [kg/m2]                            46.9            46.9 (-0.0%)
cruise speed [kt]                                161             160 (-0.5%)
cruise L/D                                      9.05            8.95 (-1.1%)
motor ratin

The pack shrinks, the aircraft gets lighter, and the generators may drop a unit. But check `binding_changes`: with sag relieved, **another** requirement takes over
sizing the battery (energy, current rating, or the SOC floor). A technology gain is worth what it buys *until the next constraint binds*.

In [3]:
binding_changes(base, cell8)

now binding (new):
    climb: generator temperature_C
    soc_after_engine_out_hover
no longer binding:
    engine-out hover: battery end voltage_V
still binding:
    bus-out hover: motor torque_Nm
    cn_beta_per_rad
    engine-out hover: generator torque_Nm
    engine-out hover: generator_gearbox power_input_W
    engine-out hover: turboshaft power_shaft_W
    hot-day hover: heat_exchanger power_heat_W
    hot-day hover: turboshaft power_shaft_W
    hover: gearbox power_input_W
    hover: propulsor power_shaft_W
    rotor_radius_m
    static_margin


## 2. The cost of redundancy

The baseline has two motor lanes per rotor, two cross-strapped buses and two isolated battery strings, plus the failure hovers that size them (bus out sizes the
motors). `redundancy=False` removes the hardware and its failure cases (the engine-out hover stays):

In [4]:
simplex = size("no redundancy", assumptions=replace(a, redundancy=False))
compare(base, simplex)
weight_changes(base, simplex)
check("redundancy costs weight", simplex.result.mass_takeoff_kg < base.result.mass_takeoff_kg - 100)

[no redundancy: sized in 44 s from 'baseline': 13,898 lb]
                                            baseline           no redundancy
take-off weight [lb]                          15,134          13,898 (-8.2%)
empty weight [lb]                             11,087          9,937 (-10.4%)
payload [lb]                                   1,984           1,984 (+0.0%)
fuel [lb]                                      2,063           1,977 (-4.2%)
battery [kWh]                                   69.5           56.8 (-18.3%)
wing area [m2]                                 23.24           21.71 (-6.6%)
span [m]                                       11.93           11.53 (-3.4%)
rotor radius [m]                               4.825           4.625 (-4.1%)
disk loading [kg/m2]                            46.9            46.9 (-0.1%)
cruise speed [kt]                                161             159 (-0.8%)
cruise L/D                                      9.05            8.88 (-1.9%)
motor rating per l

That difference is the **price of the redundancy architecture** in take-off weight, the number a safety and certification discussion needs. Where it comes from:
heavier motors (one lane must carry a rotor after a bus failure), contactors and ties, and the snowball on everything else. Try a middle ground in your turn below.

## 3. A model switch that goes the "wrong" way

Turn the thermal model off. Intuition says: fewer things modelled, fewer constraints, lighter aircraft. Run it:

In [5]:
no_thermal = size("thermal model off", assumptions=replace(a, thermal_model=False))
compare(base, no_thermal)
binding_changes(base, no_thermal)
check("switching the thermal model off makes this aircraft heavier", no_thermal.result.mass_takeoff_kg > base.result.mass_takeoff_kg)

[thermal model off: sized in 40 s from 'baseline': 16,504 lb]
                                            baseline       thermal model off
take-off weight [lb]                          15,134          16,504 (+9.1%)
empty weight [lb]                             11,087         12,307 (+11.0%)
payload [lb]                                   1,984           1,984 (+0.0%)
fuel [lb]                                      2,063           2,213 (+7.3%)
battery [kWh]                                   69.5           99.0 (+42.4%)
wing area [m2]                                 23.24           21.55 (-7.3%)
span [m]                                       11.93           11.48 (-3.7%)
rotor radius [m]                               4.825           4.604 (-4.6%)
disk loading [kg/m2]                            46.9           56.2 (+19.8%)
cruise speed [kt]                                161             169 (+5.2%)
cruise L/D                                      9.05            8.76 (-3.2%)
motor rating p

It gets **heavier**. Why: with the thermal model, a machine's power rating is a *continuous* rating and short hover and failure peaks are limited by winding
temperature, which the thermal mass absorbs (tutorial 09). Without it, the rating is a hard limit at every point, so the motors must be rated for the hover peaks,
and they jump a unit. The heat exchanger and cooling drag disappear, but that saving is smaller.

Two lessons:

- **A model switch is not a free simplification.** Every model changes which constraints exist and what they mean. Before quoting a trade, check whether it moved
  because of the physics or because of a modelling choice (tutorial 04: "keep the simplest model", so you can always make this comparison).
- `binding_changes` tells you which constraint flipped: here the machine power limits replace the temperature limits.

## 4. Your turn: a middle-ground architecture

Keep redundancy on but use **one battery string** and no string-out failure case. Edit and run:

In [6]:
my_assumptions = replace(a, count_strings_battery=1, failure_string_out=False)       # <- edit
mine = size("one battery string", assumptions=my_assumptions)
compare(base, mine, simplex)
weight_changes(base, mine)

[one battery string: sized in 58 s from 'baseline': 15,109 lb]
                                            baseline      one battery string           no redundancy
take-off weight [lb]                          15,134          15,109 (-0.2%)          13,898 (-8.2%)
empty weight [lb]                             11,087          11,064 (-0.2%)          9,937 (-10.4%)
payload [lb]                                   1,984           1,984 (+0.0%)           1,984 (+0.0%)
fuel [lb]                                      2,063           2,061 (-0.1%)           1,977 (-4.2%)
battery [kWh]                                   69.5            69.2 (-0.4%)           56.8 (-18.3%)
wing area [m2]                                 23.24           23.21 (-0.1%)           21.71 (-6.6%)
span [m]                                       11.93           11.92 (-0.1%)           11.53 (-3.4%)
rotor radius [m]                               4.825           4.821 (-0.1%)           4.625 (-4.1%)
disk loading [kg/m2]        

**Questions**

1. How much of the redundancy cost is the battery strings, and how much the lanes and buses?
2. Which failure case sizes the motors in your variant?

<details><summary>Hints</summary>

Compare your variant with both the baseline and the no-redundancy variant: the take-off weight differences split the redundancy cost. For question 2, look for
`bus-out hover` or `lane-out hover` constraints in `binding_changes(base, mine)` under "still binding". Other fields to try: `count_lanes_motor`, `count_buses`,
`failure_bus_out`, `failure_string_out_engine_out=True` (a double failure).
</details>

## What you practised

- Technology, architecture and model assumptions are all `replace(base.assumptions, ...)`.
- A technology gain lasts until the next constraint binds; read `binding_changes`.
- Redundancy has a price you can quote; model switches can move the answer either way, and you should always know which one you are looking at.

Next: **Workbook 4**, trade studies and plots.

In [7]:
check_summary()

3 of 3 checks passed
